In [4]:
import zipfile

zip_path = "/content/Plagiarism-checker-Python-master.zip"  # Replace with your ZIP filename if different

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/")

In [5]:
import os

os.listdir("/content")

['.config',
 'Plagiarism-checker-Python-master',
 'Articles.csv',
 'Plagiarism-checker-Python-master.zip',
 '__MACOSX',
 'sample_data']

In [6]:
%cd /content/Plagiarism-checker-Python-master

/content/Plagiarism-checker-Python-master


In [7]:
!ls

app.py	   image.png  juma.txt	README.md
fatma.txt  john.txt   pictures	requirements.txt


In [8]:
import sklearn
print(sklearn.__version__)

1.6.1


In [9]:
with open("app.py", "r") as f:
    print(f.read())

import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

student_files = [doc for doc in os.listdir() if doc.endswith('.txt')]
student_notes = [open(_file, encoding='utf-8').read()
                 for _file in student_files]


def vectorize(Text): return TfidfVectorizer().fit_transform(Text).toarray()
def similarity(doc1, doc2): return cosine_similarity([doc1, doc2])


vectors = vectorize(student_notes)
s_vectors = list(zip(student_files, vectors))
plagiarism_results = set()


def check_plagiarism():
    global s_vectors
    for student_a, text_vector_a in s_vectors:
        new_vectors = s_vectors.copy()
        current_index = new_vectors.index((student_a, text_vector_a))
        del new_vectors[current_index]
        for student_b, text_vector_b in new_vectors:
            sim_score = similarity(text_vector_a, text_vector_b)[0][1]
            student_pair = sorted((student_a, student_b))
            score = (stude

In [ ]:
!pip install nltk spacy rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 34.2 MB/s eta 0:00:00


In [10]:
import nltk

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [15]:
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Initialize
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Tokenize
    words = word_tokenize(text)

    cleaned_words = []

    for word in words:
        if word not in stop_words and len(word) > 2:
            # Lemmatize as a verb
            cleaned_words.append(
                lemmatizer.lemmatize(word, pos="v")
            )

    # Return cleaned text
    return " ".join(cleaned_words)

In [17]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [18]:
sample = "Python is AMAZING!! I love learning Python because it is powerful."

print(preprocess(sample))

python amaze love learn python powerful


In [19]:
import os

# Only read student documents
student_files = [
    file for file in os.listdir()
    if file.endswith(".txt") and file != "requirements.txt"
]

student_notes = []

for file in student_files:
    with open(file, "r", encoding="utf-8") as f:
        text = f.read()
        cleaned_text = preprocess(text)
        student_notes.append(cleaned_text)

print(student_files)

['john.txt', 'juma.txt', 'fatma.txt']


In [20]:
for i in range(len(student_files)):
    print("="*50)
    print(student_files[i])
    print(student_notes[i])

john.txt
life find money spend luxury stuff coz life kinda short trust
juma.txt
life find money use things make happy coz life kinda short
fatma.txt
life best try find work take time try pursue skills


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(student_notes)

print(tfidf_matrix.shape)

(3, 21)


In [22]:
print(tfidf_matrix.toarray())

[[0.         0.26792143 0.20806489 0.         0.26792143 0.41612978
  0.35228448 0.         0.26792143 0.         0.26792143 0.
  0.35228448 0.35228448 0.         0.         0.         0.35228448
  0.         0.         0.        ]
 [0.         0.26792143 0.20806489 0.35228448 0.26792143 0.41612978
  0.         0.35228448 0.26792143 0.         0.26792143 0.
  0.         0.         0.         0.35228448 0.         0.
  0.         0.35228448 0.        ]
 [0.30574243 0.         0.1805764  0.         0.         0.1805764
  0.         0.         0.         0.30574243 0.         0.30574243
  0.         0.         0.30574243 0.         0.30574243 0.
  0.61148486 0.         0.30574243]]


In [23]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

print(similarity_matrix)

[[1.         0.50358257 0.11271483]
 [0.50358257 1.         0.11271483]
 [0.11271483 0.11271483 1.        ]]


In [24]:
def plagiarism_status(score):

    if score >= 0.80:
        return "🔴 High Plagiarism"

    elif score >= 0.60:
        return "🟠 Possible Plagiarism"

    elif score >= 0.40:
        return "🟡 Needs Review"

    else:
        return "🟢 Safe"

In [27]:
!pip install rapidfuzz
from rapidfuzz import fuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 34.9 MB/s eta 0:00:00


In [28]:
def fuzzy_similarity(text1, text2):
    return fuzz.ratio(text1, text2) / 100

In [32]:
print(fuzzy_similarity(student_notes[0], student_notes[1]))

0.7058823529411764


In [29]:
print(fuzzy_similarity(student_notes[1], student_notes[2]))

0.47706422018348627


In [34]:
def final_similarity_score(cosine, fuzzy):

    # TF-IDF is more reliable than fuzzy matching
    return round((0.7 * cosine) + (0.3 * fuzzy), 2)

In [35]:
for i in range(len(student_files)):
    for j in range(i + 1, len(student_files)):

        cosine_score = similarity_matrix[i][j]
        fuzzy_score = fuzzy_similarity(student_notes[i], student_notes[j])

        final_score = final_similarity_score(cosine_score, fuzzy_score)

        print("=" * 70)
        print(f"Document 1        : {student_files[i]}")
        print(f"Document 2        : {student_files[j]}")
        print(f"Cosine Similarity : {cosine_score:.2f}")
        print(f"Fuzzy Similarity  : {fuzzy_score:.2f}")
        print(f"Final Score       : {final_score:.2f}")
        print(f"Status            : {plagiarism_status(final_score)}")

Document 1        : john.txt
Document 2        : juma.txt
Cosine Similarity : 0.50
Fuzzy Similarity  : 0.71
Final Score       : 0.56
Status            : 🟡 Needs Review
Document 1        : john.txt
Document 2        : fatma.txt
Cosine Similarity : 0.11
Fuzzy Similarity  : 0.45
Final Score       : 0.21
Status            : 🟢 Safe
Document 1        : juma.txt
Document 2        : fatma.txt
Cosine Similarity : 0.11
Fuzzy Similarity  : 0.48
Final Score       : 0.22
Status            : 🟢 Safe


In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    lowercase=True
)

tfidf_matrix = vectorizer.fit_transform(student_notes)

print(tfidf_matrix.shape)

(3, 45)


In [37]:
feature_names = vectorizer.get_feature_names_out()

print("Total Features:", len(feature_names))
print(feature_names)

Total Features: 45
['best' 'best try' 'coz' 'coz life' 'find' 'find money' 'find work'
 'happy' 'happy coz' 'kinda' 'kinda short' 'life' 'life best' 'life find'
 'life kinda' 'luxury' 'luxury stuff' 'make' 'make happy' 'money'
 'money spend' 'money use' 'pursue' 'pursue skills' 'short' 'short trust'
 'skills' 'spend' 'spend luxury' 'stuff' 'stuff coz' 'take' 'take time'
 'things' 'things make' 'time' 'time try' 'trust' 'try' 'try find'
 'try pursue' 'use' 'use things' 'work' 'work take']


In [38]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

In [39]:
import pandas as pd

results = []

for i in range(len(student_files)):
    for j in range(i + 1, len(student_files)):

        cosine_score = similarity_matrix[i][j]
        fuzzy_score = fuzzy_similarity(student_notes[i], student_notes[j])
        final_score = final_similarity_score(cosine_score, fuzzy_score)

        results.append({
            "Document 1": student_files[i],
            "Document 2": student_files[j],
            "Cosine Similarity": round(cosine_score, 2),
            "Fuzzy Similarity": round(fuzzy_score, 2),
            "Final Score": round(final_score, 2),
            "Status": plagiarism_status(final_score)
        })

df = pd.DataFrame(results)

df

,Document 1,Document 2,Cosine Similarity,Fuzzy Similarity,Final Score,Status
0,john.txt,juma.txt,0.44,0.71,0.52,🟡 Needs Review
1,john.txt,fatma.txt,0.06,0.45,0.18,🟢 Safe
2,juma.txt,fatma.txt,0.06,0.48,0.18,🟢 Safe


In [40]:
df.to_csv("plagiarism_report.csv", index=False)

print("Report generated successfully!")

Report generated successfully!


In [41]:
print(df)

  Document 1 Document 2  Cosine Similarity  Fuzzy Similarity  Final Score  \
0   john.txt   juma.txt               0.44              0.71         0.52   
1   john.txt  fatma.txt               0.06              0.45         0.18   
2   juma.txt  fatma.txt               0.06              0.48         0.18   

           Status  
0  🟡 Needs Review  
1          🟢 Safe  
2          🟢 Safe  


In [42]:
!ls /content

Articles.csv  Plagiarism-checker-Python-master	    sample_data
__MACOSX      Plagiarism-checker-Python-master.zip


In [44]:
import pandas as pd

df = pd.read_csv("/content/Articles.csv", encoding="latin1")

In [45]:
print(df.columns)
print(df.info())
print(df.head())

Index(['Article', 'Date', 'Heading', 'NewsType'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2692 entries, 0 to 2691
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Article   2692 non-null   object
 1   Date      2692 non-null   object
 2   Heading   2692 non-null   object
 3   NewsType  2692 non-null   object
dtypes: object(4)
memory usage: 84.3+ KB
None
                                             Article      Date  \
0  KARACHI: The Sindh government has decided to b...  1/1/2015   
1  HONG KONG: Asian markets started 2015 on an up...  1/2/2015   
2  HONG KONG:  Hong Kong shares opened 0.66 perce...  1/5/2015   
3  HONG KONG: Asian markets tumbled Tuesday follo...  1/6/2015   
4  NEW YORK: US oil prices Monday slipped below $...  1/6/2015   

                                             Heading  NewsType  
0  sindh govt decides to cut public transport far...  business  
1                    asia s

In [46]:
news_df = df.sample(
    n=50,
    random_state=42
).reset_index(drop=True)

print(news_df.shape)

(50, 4)


In [47]:
news_articles = news_df["Article"].tolist()

In [48]:
processed_articles = []

for article in news_articles:
    processed_articles.append(preprocess(article))

print(processed_articles[0])

strongkathmandu yearold dutch climber die descend summit everest first perish year worlds highest mountain officials nepal say saturdaystrongeric ary arnold among climbers reach metre feet summit friday die later day come highaltitude slop know death zone prevail thin air tourism department official gyanendra shrestha saidmingma sherpa seven summit trek company organise arnolds expedition say client complain weakness descend metres feet probably die altitude sicknesswe try get touch family insurance company body still mountain saida nepali sherpa fix rope nearby lhotse worlds fourth highest peak metres feet fell death week climbers reach top everest monthan earthquake last year kill least people everest base camp situate metres feet altitude force hundreds climbers abandon expeditions quake worst nepals record history kill people across himalayan nation


In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1,2),
    max_df=0.90,
    min_df=2,
    sublinear_tf=True
)

news_matrix = vectorizer.fit_transform(processed_articles)

In [50]:
from sklearn.metrics.pairwise import cosine_similarity

news_similarity = cosine_similarity(news_matrix)

print(news_similarity.shape)

(50, 50)


In [51]:
results = []

for i in range(len(news_articles)):
    for j in range(i + 1, len(news_articles)):

        cosine_score = news_similarity[i][j]

        fuzzy_score = fuzzy_similarity(
            processed_articles[i],
            processed_articles[j]
        )

        final_score = final_similarity_score(
            cosine_score,
            fuzzy_score
        )

        results.append({
            "Article 1": i,
            "Article 2": j,
            "Heading 1": news_df.iloc[i]["Heading"],
            "Heading 2": news_df.iloc[j]["Heading"],
            "Cosine Similarity": round(cosine_score, 2),
            "Fuzzy Similarity": round(fuzzy_score, 2),
            "Final Score": round(final_score, 2),
            "Status": plagiarism_status(final_score)
        })

In [52]:
report = pd.DataFrame(results)

report = report.sort_values(
    by="Final Score",
    ascending=False
).reset_index(drop=True)

report.head(10)

,Article 1,Article 2,Heading 1,Heading 2,Cosine Similarity,Fuzzy Similarity,Final Score,Status
0,3,21,Asian shares edge lower as Fed rate talk reviv,Asian shares rise dollar nurses losses after j...,0.51,0.47,0.49,🟡 Needs Review
1,2,13,oil prices up in asi,oil drops after yemen inspired gai,0.44,0.43,0.43,🟡 Needs Review
2,38,46,Cook ready for Pakistan after Sri Lanka su,England call up uncapped trio for Sri Lanka T20,0.38,0.43,0.39,🟢 Safe
3,40,45,New Zealand record dramatic win over Aussi,Pakistan win toss opt to field in T20 series d...,0.38,0.41,0.39,🟢 Safe
4,3,29,Asian shares edge lower as Fed rate talk reviv,Gold slips Brexit tensions ease stocks rally,0.36,0.44,0.39,🟢 Safe
5,21,29,Asian shares rise dollar nurses losses after j...,Gold slips Brexit tensions ease stocks rally,0.36,0.44,0.38,🟢 Safe
6,24,36,Economic Survey Pakistan misses GDP targ,punjab govt presents rs 1.45tr budg,0.35,0.45,0.38,🟢 Safe
7,13,32,oil drops after yemen inspired gai,Iran says will resist curbs on oil output as p...,0.34,0.43,0.37,🟢 Safe
8,28,45,Boult strikes hurt Zimbabwe after New Zealand ...,Pakistan win toss opt to field in T20 series d...,0.34,0.40,0.36,🟢 Safe
9,28,40,Boult strikes hurt Zimbabwe after New Zealand ...,New Zealand record dramatic win over Aussi,0.30,0.43,0.34,🟢 Safe


In [53]:
report.to_csv(
    "news_plagiarism_report.csv",
    index=False
)

print("Report Saved Successfully!")

Report Saved Successfully!


In [54]:
from google.colab import files

files.download("plagiarism_report.csv")
files.download("news_plagiarism_report.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>